# 12 — Reporte del EDA: congestión de la justicia no penal en 2023
Alcance: solo el estrato `proceso` (materias no penales). Los cuadros penales no publican
`resueltas` y no admiten las fórmulas CEJA. Ninguna cifra de acá es "todo el sistema judicial".

In [ ]:
%run -i modulos/comun.ipynb
import numpy as np
import matplotlib.pyplot as plt

OUT_TABLES = REPORTS / "tables"
OUT_FIG = REPORTS / "figures"
OUT_TABLES.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(CURATED / "features" / "dataset_analitico_curado.parquet")
proc = df[df["tipo_elemento_analitico"] == "proceso"].copy()
print(len(proc), "procesos de", len(df), "filas")


def agregar_tasas(t):
    t["cr_ponderado"] = t["resueltas"] / t["ingresos_totales"].replace(0, np.nan)
    t["congestion_ponderada"] = t["atendidas"] / t["resueltas"].replace(0, np.nan)
    t["duracion_dias_ponderada"] = (t["pendientes_fin"] / t["resueltas"].replace(0, np.nan)) * 365
    return t


COMUNES = {
    "total_procesos": ("tipo_proceso", "count"),
    "ingresos_totales": ("ingresos_totales", "sum"),
    "nuevas_ingresadas": ("nuevas_ingresadas", "sum"),
    "atendidas": ("atendidas", "sum"),
    "resueltas": ("resueltas", "sum"),
    "pendientes_fin": ("pendientes_fin", "sum"),
}
MEDIANAS = {
    "cr_mediana": ("tasa_resolucion", "median"),
    "congestion_mediana": ("tasa_congestion", "median"),
    "duracion_dias_mediana": ("duracion_estimada_dias", "median"),
}

## Balance general

In [ ]:
atendidas_universo = int(df["atendidas"].sum())
total_ingresos = int(proc["ingresos_totales"].sum())
total_atendidas = int(proc["atendidas"].sum())
total_resueltas = int(proc["resueltas"].sum())
total_pendientes = int(proc["pendientes_fin"].sum())
print("cobertura de atendidas: {:.1%}".format(total_atendidas / atendidas_universo))
print("ingresadas", total_ingresos, "| atendidas", total_atendidas, "| resueltas", total_resueltas, "| pendientes", total_pendientes)
print("tasa de resolución global: {:.2%}".format(total_resueltas / total_ingresos))
print("tasa de congestión global: {:.2f}".format(total_atendidas / total_resueltas))
print("materias fuera del reporte:", sorted(df.loc[df["tipo_elemento_analitico"] != "proceso", "materia_homologada"].dropna().unique()))

## Por departamento
Cuidado con `causas_por_funcionario`: el numerador es solo lo no penal y el denominador todo el personal.

In [ ]:
agregaciones = dict(COMUNES)
agregaciones["personal_items"] = ("personal_items_total", "first")
agregaciones["juzgados_publicados"] = ("recurso_juzgados_publicados", "sum")
agregaciones.update(MEDIANAS)
deptos = agregar_tasas(proc.groupby("departamento_derivado").agg(**agregaciones).reset_index())
deptos["causas_por_funcionario"] = deptos["atendidas"] / deptos["personal_items"].replace(0, np.nan)
deptos = deptos.sort_values(by="atendidas", ascending=False)
deptos.to_csv(OUT_TABLES / "resumen_congestion_departamental.csv", index=False, encoding="utf-8")
deptos

## Por materia

In [ ]:
agregaciones = dict(COMUNES)
agregaciones.update(MEDIANAS)
agregaciones["n_outliers_carga"] = ("es_outlier_carga_iqr", "sum")
agregaciones["n_outliers_congestion"] = ("es_outlier_congestion_iqr", "sum")
materias = agregar_tasas(proc.groupby("materia_homologada").agg(**agregaciones).reset_index())
materias = materias.sort_values(by="atendidas", ascending=False)
materias.to_csv(OUT_TABLES / "resumen_congestion_materia.csv", index=False, encoding="utf-8")
materias

## Capitales vs. provincias

In [ ]:
agregaciones = dict(COMUNES)
agregaciones.update(MEDIANAS)
ambitos = agregar_tasas(proc.groupby("ambito").agg(**agregaciones).reset_index())
ambitos.to_csv(OUT_TABLES / "resumen_congestion_ambito.csv", index=False, encoding="utf-8")
ambitos

## Los 20 tipos de proceso con más causas pendientes

In [ ]:
top = proc.groupby(["materia_homologada", "tipo_proceso"]).agg(
    atendidas_total=("atendidas", "sum"),
    resueltas_total=("resueltas", "sum"),
    pendientes_total=("pendientes_fin", "sum"),
    duracion_media_dias=("duracion_estimada_dias", "mean"),
    congestion_media=("tasa_congestion", "mean"),
).reset_index()
top["congestion_ponderada"] = top["atendidas_total"] / top["resueltas_total"].replace(0, np.nan)
top["duracion_ponderada_dias"] = (top["pendientes_total"] / top["resueltas_total"].replace(0, np.nan)) * 365
top20 = top.sort_values(by="pendientes_total", ascending=False).head(20)
top20.to_csv(OUT_TABLES / "top_procesos_congestionados.csv", index=False, encoding="utf-8")
top20

In [ ]:
t = top20.iloc[::-1]
plt.figure(figsize=(10, 7))
plt.barh(t["tipo_proceso"].str[:45] + " (" + t["materia_homologada"].str[:12] + ")", t["pendientes_total"], color="tab:red")
plt.xlabel("causas pendientes al cierre de 2023")
plt.title("Top 20 tipos de proceso por stock pendiente")
plt.tight_layout()
plt.show()

## Mapa de calor: congestión mediana (winsorizada) por departamento y materia

In [ ]:
pivot = proc.pivot_table(index="departamento_derivado", columns="materia_homologada", values="tasa_congestion_winsorizada", aggfunc="median")
fig, ax = plt.subplots(figsize=(12, 7))
im = ax.imshow(pivot.values, cmap="YlOrRd", aspect="auto")
ax.set_xticks(np.arange(len(pivot.columns)))
ax.set_yticks(np.arange(len(pivot.index)))
ax.set_xticklabels([c[:20] for c in pivot.columns], rotation=30, ha="right", fontsize=9)
ax.set_yticklabels(pivot.index, fontsize=10)
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        valor = pivot.values[i, j]
        if not np.isnan(valor):
            if valor > 2.5:
                color = "white"
            else:
                color = "black"
            ax.text(j, i, "{:.2f}".format(valor), ha="center", va="center", color=color, fontsize=9, fontweight="bold")
barra = fig.colorbar(im, ax=ax)
barra.ax.set_ylabel("tasa de congestión mediana (winsorizada)", rotation=-90, va="bottom")
ax.set_title("Tasa de congestión por departamento y materia (2023)")
plt.tight_layout()
plt.savefig(OUT_FIG / "03_heatmap_congestion_departamento_materia.png", dpi=200)
plt.show()

## Personal vs. causas atendidas por departamento (tamaño = congestión, color = tasa de resolución)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
puntos = ax.scatter(deptos["personal_items"], deptos["atendidas"], s=deptos["congestion_ponderada"] * 120,
                    c=deptos["cr_ponderado"], cmap="coolwarm_r", alpha=0.85, edgecolors="black", linewidth=1.2)
for i, r in deptos.iterrows():
    ax.annotate(r["departamento_derivado"], (r["personal_items"], r["atendidas"]), xytext=(6, 4), textcoords="offset points", fontweight="bold")
ax.set_xlabel("ítems de personal del distrito judicial")
ax.set_ylabel("causas atendidas (no penales)")
ax.grid(True, linestyle="--", alpha=0.5)
barra = plt.colorbar(puntos, ax=ax)
barra.ax.set_ylabel("tasa de resolución ponderada", rotation=-90, va="bottom")
ax.set_title("Personal judicial y volumen de causas (2023)")
plt.tight_layout()
plt.savefig(OUT_FIG / "04_dispersion_recursos_vs_resolucion.png", dpi=200)
plt.show()

## Capitales vs. provincias en una imagen

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(13, 3.5))
indicadores = [("cr_ponderado", "Tasa de resolución"), ("congestion_ponderada", "Tasa de congestión"), ("duracion_dias_ponderada", "Duración (días)")]
for k in range(3):
    col, titulo = indicadores[k]
    ejes[k].bar(ambitos["ambito"], ambitos[col], color=["tab:blue", "tab:orange"])
    ejes[k].set_title(titulo)
plt.tight_layout()
plt.savefig(OUT_FIG / "05_capital_vs_provincia.png", dpi=150)
plt.show()